<a href="https://colab.research.google.com/github/ritesh586/customer-churn-prediction/blob/main/05_going_modular.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import os
import torch
from torchvision import transforms
import requests
import zipfile
from pathlib import Path



In [6]:
data_path=Path("data/")
image_path=data_path/"pizza_sushi_steak"

if image_path.is_dir():
  print("Directory already exists")
else:
  print("Creating directory")
  image_path.mkdir(parents=True,exist_ok=True)

with open(data_path / "pizza_steak_sushi.zip", "wb") as f:
  request= requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip")
  print("Downloading images")
  f.write(request.content)

with zipfile.ZipFile("data/pizza_steak_sushi.zip","r") as zip_ref:
  print("Unzipping images")
  zip_ref.extractall(image_path)

os.remove("data/pizza_steak_sushi.zip")

Directory already exists
Unzipping images


In [7]:
import os
os.makedirs("going_modular", exist_ok=True)

with open("going_modular/data_setup.py", "w") as f:
  f.write("""import os

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

Num_workers=os.cpu_count()

def create_dataloaders(train_dir,test_dir
                       , batch_size
                       ,transform
                       ,num_workers=Num_workers):
  train_data=datasets.ImageFolder(train_dir,transform=transform)
  test_data=datasets.ImageFolder(test_dir,transform=transform)

  class_names=train_data.classes

  train_dataloader=DataLoader(train_data,batch_size=batch_size,shuffle=True,
                              num_workers=num_workers,pin_memory=True)
  test_dataloader=DataLoader(test_data,batch_size=batch_size,shuffle=True,
                              num_workers=num_workers,pin_memory=True)
  return train_dataloader,test_dataloader ,class_names
""")

In [8]:
from going_modular import data_setup
train_dataloader, test_dataloader ,class_names=data_setup.create_dataloaders(image_path,image_path,batch_size=32,transform=transforms.ToTensor())

In [9]:
%%writefile going_modular/model_builder.py

import torch
from torch import nn
class TinyVGG(nn.Module):
  def __init__(self,input_shape,hidden_layer,output_shape):
    super().__init__()
    self.conv_block_1=nn.Sequential(
        nn.Conv2d(
            input_shape,hidden_layer,kernel_size=3,stride=1,padding=0
        ),
        nn.ReLU(),
        nn.Conv2d(
            hidden_layer,hidden_layer,kernel_size=3,stride=1,padding=0
        ),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2)
    )
    self.conv_block_2=nn.Sequential(
        nn.Conv2d(
            hidden_layer,hidden_layer,kernel_size=3,stride=1,padding=0
        ),
        nn.ReLU(),
        nn.Conv2d(
            hidden_layer,hidden_layer,kernel_size=3,stride=1,padding=0
        ),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2)
    )
    self.classifier=nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=hidden_layer*13*13,
                    out_features=output_shape)
    )
    def forward(self,x):
      x=self.conv_block_1(x)
      x=self.conv_block_2(x)
      x=self.classifier(x)
      return x


Writing going_modular/model_builder.py


In [10]:
%%writefile going_modular/engine.py
import torch

from tqdm.auto import tqdm
from typing import Dict, List, Tuple

def train_step(model,train_dataloader,loss_fn,optimizer,device):
  model.train()
  train_loss,train_acc=0,0
  for batch, (X,y) in enumerate(train_dataloader):
    y_preds=model(X)
    loss=loss_fn(y_preds,y)
    train_loss+=loss.item()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    y_pred_class=torch.argmax(torch.softmax(y_preds,dim=1),dim=1)
    train_acc+=(y_pred_class==y).sum().item()/len(y_pred_class)
  train_loss=train_loss/len(train_dataloader)
  train_acc=train_acc/len(train_dataloader)
  return train_loss,train_acc

def test_step(model,test_dataloader,loss_fn,device):
  model.eval()
  test_loss,test_acc=0,0
  with torch.inference_mode():
    for batch,(X,y) in enumerate(test_dataloader):
      test_preds=model(X)
      loss=loss_fn(test_preds,y)
      test_loss+=loss.item()
      test_pred_labels=test_preds.argmax(dim=1
                                      )
      test_acc=(test_pred_labels==y).sum().item()/len(test_pred_labels)
  test_loss=test_loss/len(test_dataloader)
  test_acc=test_acc/len(test_dataloader)
  return test_loss,test_acc

def train(model,train_dataloader,test_dataloader,optimizer,loss_fn,epochs,device):
  results={
      "train_loss":[],
      "train_acc":[],
      "test_loss":[],
      "test_acc":[]
  }

  for epoch in tqdm(range(epochs)):
    train_loss,train_acc=train_step(model=model,train_dataloader=train_dataloader,loss_fn=loss_fn,optimizer=optimizer,device=device)
    test_loss, test_acc = test_step(model=model,
          dataloader=test_dataloader,
          loss_fn=loss_fn,
          device=device)
    print(
          f"Epoch: {epoch+1} | "
          f"train_loss: {train_loss:.4f} | "
          f"train_acc: {train_acc:.4f} | "
          f"test_loss: {test_loss:.4f} | "
          f"test_acc: {test_acc:.4f}"
      )

      # Update results dictionary
    results["train_loss"].append(train_loss)
    results["train_acc"].append(train_acc)
    results["test_loss"].append(test_loss)
    results["test_acc"].append(test_acc)

  # Return the filled results at the end of the epochs
  return results


Writing going_modular/engine.py


In [11]:
%%writefile going_modular/utils.py
import torch
from pathlib import Path

def save_model(model: torch.nn.Module,
               target_dir: str,
               model_name: str):
  target_dir_path = Path(target_dir)
  target_dir_path.mkdir(parents=True,
                        exist_ok=True)
  assert model_name.endswith(".pth") or model_name.endswith(".pt"), "model_name should end with '.pt' or '.pth'"
  model_save_path = target_dir_path / model_name

  print(f"[INFO] Saving model to: {model_save_path}")
  torch.save(obj=model.state_dict(),
             f=model_save_path)

Writing going_modular/utils.py


In [12]:
%%writefile going_modular/train.py
import os
import torch
import data_setup, engine, model_builder, utils

from torchvision import transforms
NUM_EPOCHS = 5
BATCH_SIZE = 32
HIDDEN_UNITS = 10
LEARNING_RATE = 0.001

# Setup directories
train_dir = "data/pizza_steak_sushi/train"
test_dir = "data/pizza_steak_sushi/test"

# Setup target device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Create transforms
data_transform = transforms.Compose([
  transforms.Resize((64, 64)),
  transforms.ToTensor()
])

# Create DataLoaders with help from data_setup.py
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=data_transform,
    batch_size=BATCH_SIZE
)

# Create model with help from model_builder.py
model = model_builder.TinyVGG(
    input_shape=3,
    hidden_units=HIDDEN_UNITS,
    output_shape=len(class_names)
).to(device)

# Set loss and optimizer
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),
                             lr=LEARNING_RATE)

# Start training with help from engine.py
engine.train(model=model,
             train_dataloader=train_dataloader,
             test_dataloader=test_dataloader,
             loss_fn=loss_fn,
             optimizer=optimizer,
             epochs=NUM_EPOCHS,
             device=device)

# Save the model with help from utils.py
utils.save_model(model=model,
                 target_dir="models",
                 model_name="05_going_modular_script_mode_tinyvgg_model.pth")

Writing going_modular/train.py
